In [2]:
import timm

model1 = timm.create_model('efficientnet_b3', pretrained=True)
print("✅ EfficientNet B3 완료")

model2 = timm.create_model('efficientnet_b4', pretrained=True)
print("✅ EfficientNet B4 완료")

model3 = timm.create_model('resnet50', pretrained=True)
print("✅ ResNet50 완료")

model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

✅ EfficientNet B3 완료


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

✅ EfficientNet B4 완료


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

✅ ResNet50 완료


In [3]:
from monai.networks.nets import DenseNet121, EfficientNetBN

model4 = DenseNet121(
    spatial_dims=2,
    in_channels=1,   # 초음파 흑백
    out_channels=2   # 양성/악성
)
print("✅ MONAI DenseNet121 완료")

model5 = EfficientNetBN(
    model_name="efficientnet-b4",
    pretrained=True,
    spatial_dims=2,
    in_channels=1,
    num_classes=2
)
print("✅ MONAI EfficientNet-B4 완료")

✅ MONAI DenseNet121 완료


Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b4-6ed6700e.pth" to /home/ubuntu/.cache/torch/hub/checkpoints/efficientnet-b4-6ed6700e.pth
100%|█████████████████████████████████████████████████████████████████████████| 74.4M/74.4M [00:01<00:00, 51.7MB/s]

✅ MONAI EfficientNet-B4 완료


In [5]:
code = '''
import os
import pandas as pd
import numpy as np
import pydicom
import cv2
import torch
from pathlib import Path

# ================================
# 1. 데이터 구조 파악
# ================================
data_root = "/안심존데이터/"

for root, dirs, files in os.walk(data_root):
    level = root.replace(data_root, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        subindent = " " * 2 * (level + 1)
        for file in files[:5]:
            print(f"{subindent}{file}")

# ================================
# 2. EMR 메타데이터 분석
# ================================
def load_emr(path):
    import shutil
    shutil.copytree("/안심존데이터/EMR/", "/home/ubuntu/project/data/emr/")
    df = pd.read_csv(path)
    print(df.shape)
    print(df.dtypes)
    print(df.isnull().sum())
    print(df.describe())
    return df

# ================================
# 3. 초음파 이미지 로드
# ================================
def load_ultrasound(img_dir):
    img_paths = list(Path(img_dir).glob("**/*.dcm"))
    print(f"총 이미지 수: {len(img_paths)}")
    
    # DICOM 읽기
    ds = pydicom.dcmread(str(img_paths[0]))
    img = ds.pixel_array
    print(f"Shape: {img.shape}, dtype: {img.dtype}")
    return img_paths

# ================================
# 4. GPU 확인
# ================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
'''

# 파일로 저장
with open("/home/ubuntu/project/src/analysis.py", "w") as f:
    f.write(code)

print("✅ analysis.py 저장 완료!")

✅ analysis.py 저장 완료!


In [6]:
import os
print(os.listdir("/home/ubuntu/project/src/"))

['analysis.py']


In [7]:
code_explore = '''
import os
import json
import pandas as pd
from pathlib import Path

data_root = "/안심존데이터/"  # 마운트 후 실제 경로 확인

# 폴더 구조 파악
for root, dirs, files in os.walk(data_root):
    level = root.replace(data_root, "").count(os.sep)
    if level > 2:
        continue
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for file in files[:3]:
        print(f"{subindent}{file}")
'''

with open("/home/ubuntu/project/src/01_explore.py", "w") as f:
    f.write(code_explore)
print("✅ 01_explore.py 저장")

✅ 01_explore.py 저장


In [8]:
code_ultrasound = '''
import cv2
import numpy as np
import json
from pathlib import Path
import shutil

# 원본 → 작업폴더 복사
src = "/안심존데이터/초음파/"
dst = "/home/ubuntu/project/data/ultrasound/"
# shutil.copytree(src, dst)  # 마운트 후 주석 해제

img_paths = list(Path(dst).glob("**/*.png"))
print(f"총 초음파 이미지: {len(img_paths)}개")

def preprocess_ultrasound(img_path, size=(224, 224)):
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, size)
    img = img / 255.0  # 정규화
    return img

# JSON 라벨 읽기
def load_label(json_path):
    with open(json_path) as f:
        data = json.load(f)
    return data
'''

with open("/home/ubuntu/project/src/02_ultrasound.py", "w") as f:
    f.write(code_ultrasound)
print("✅ 02_ultrasound.py 저장")

✅ 02_ultrasound.py 저장


In [9]:
code_emr = '''
import pandas as pd
import numpy as np
import shutil

# 원본 → 작업폴더 복사
src = "/안심존데이터/메타데이터/"
dst = "/home/ubuntu/project/data/metadata/"
# shutil.copytree(src, dst)  # 마운트 후 주석 해제

def load_metadata(path):
    df = pd.read_csv(path)  # 또는 read_excel
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    print("결측치:", df.isnull().sum())
    print(df.describe())
    return df

# 주요 컬럼 (난소암 기준)
# CA-125, 나이, 병기, 조직학적유형, 악성여부
def preprocess_emr(df):
    df = df.dropna(subset=["label"])  # 라벨 없는 행 제거
    df = pd.get_dummies(df, columns=["조직학적유형"])  # 범주형 인코딩
    return df
'''

with open("/home/ubuntu/project/src/03_emr.py", "w") as f:
    f.write(code_emr)
print("✅ 03_emr.py 저장")

✅ 03_emr.py 저장


In [10]:
import os
print(os.listdir("/home/ubuntu/project/src/"))

['02_ultrasound.py', '03_emr.py', '01_explore.py', 'analysis.py']


In [1]:
from ultralytics import YOLO

# YOLOv8 모델 다운로드 (의료영상에 많이 씀)
model_n = YOLO('yolov8n.pt')  # nano  - 가장 빠름
print("✅ YOLOv8n 완료")

model_s = YOLO('yolov8s.pt')  # small
print("✅ YOLOv8s 완료")

model_m = YOLO('yolov8m.pt')  # medium - 초음파에 추천
print("✅ YOLOv8m 완료")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/ubuntu/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ YOLOv8n 완료
✅ YOLOv8s 완료
✅ YOLOv8m 완료


In [2]:
import shutil
import os

yolo_dir = "/home/ubuntu/project/models/yolo"
os.makedirs(yolo_dir, exist_ok=True)

# ultralytics가 자동 저장한 pt 파일 복사
for pt in ['yolov8n.pt', 'yolov8s.pt', 'yolov8m.pt']:
    src = os.path.expanduser(f"~/.cache/ultralytics/{pt}")
    if not os.path.exists(src):
        src = pt  # 현재 디렉토리에 있을 경우
    try:
        shutil.copy(src, f"{yolo_dir}/{pt}")
        print(f"✅ {pt} 저장완료")
    except:
        print(f"⚠️ {pt} 경로 확인 필요")

print(os.listdir(yolo_dir))

✅ yolov8n.pt 저장완료
✅ yolov8s.pt 저장완료
✅ yolov8m.pt 저장완료
['yolov8s.pt', 'yolov8m.pt', 'yolov8n.pt']


In [3]:
code_yolo = '''
from ultralytics import YOLO
import os

# 저장된 모델 로드
model = YOLO("/home/ubuntu/project/models/yolo/yolov8m.pt")

# 학습 설정 (마운트 후 데이터 경로 수정)
results = model.train(
    data="/home/ubuntu/project/src/ovarian_data.yaml",  # 마운트 후 작성
    epochs=100,
    imgsz=640,
    batch=16,          # V100 32GB라 넉넉
    device=0,          # GPU 사용
    project="/home/ubuntu/project/results",
    name="ovarian_yolo",
    patience=20,       # early stopping
    save=True,
    plots=True
)

# 검증
metrics = model.val()
print(metrics)

# 예측
pred = model.predict(
    source="/home/ubuntu/project/data/ultrasound/",
    save=True,
    conf=0.25
)
'''

with open("/home/ubuntu/project/src/04_yolo_train.py", "w") as f:
    f.write(code_yolo)
print("✅ 04_yolo_train.py 저장")

✅ 04_yolo_train.py 저장


In [4]:
yaml_content = """
# 마운트 후 경로 수정 필요
path: /home/ubuntu/project/data/ultrasound
train: images/train
val: images/val
test: images/test

nc: 2  # 클래스 수 (양성/악성)
names:
  0: benign    # 양성
  1: malignant # 악성
"""

with open("/home/ubuntu/project/src/ovarian_data.yaml", "w") as f:
    f.write(yaml_content)
print("✅ ovarian_data.yaml 저장")

✅ ovarian_data.yaml 저장


In [5]:
import os
print(os.listdir("/home/ubuntu/project/src/"))
print(os.listdir("/home/ubuntu/project/models/"))

['02_ultrasound.py', '04_yolo_train.py', '03_emr.py', '01_explore.py', 'ovarian_data.yaml', 'analysis.py']
['pretrained', 'yolo']


In [6]:
import numpy as np
import cv2
import os

# 샘플 이미지 생성 (실제 초음파처럼 흑백)
sample_dir = "/home/ubuntu/project/data/sample"
os.makedirs(f"{sample_dir}/images", exist_ok=True)
os.makedirs(f"{sample_dir}/labels", exist_ok=True)

for i in range(10):
    # 224x224 흑백 노이즈 이미지 (초음파 흉내)
    img = np.random.randint(0, 255, (224, 224), dtype=np.uint8)
    img = cv2.GaussianBlur(img, (15, 15), 0)
    cv2.imwrite(f"{sample_dir}/images/sample_{i}.png", img)

print(f"✅ 샘플 이미지 10개 생성")
print(os.listdir(f"{sample_dir}/images"))

✅ 샘플 이미지 10개 생성
['sample_1.png', 'sample_3.png', 'sample_6.png', 'sample_5.png', 'sample_8.png', 'sample_4.png', 'sample_2.png', 'sample_7.png', 'sample_9.png', 'sample_0.png']


In [7]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'patient_id': range(10),
    'age': np.random.randint(30, 70, 10),
    'ca125': np.random.uniform(10, 500, 10),  # 암표지자
    'tumor_size': np.random.uniform(1, 10, 10),
    'label': np.random.randint(0, 2, 10)  # 0:양성, 1:악성
})

df.to_csv("/home/ubuntu/project/data/sample/emr_sample.csv", index=False)
print("✅ 샘플 EMR 생성")
print(df)

✅ 샘플 EMR 생성
   patient_id  age       ca125  tumor_size  label
0           0   55   77.748798    5.217806      1
1           1   61  440.213504    1.536443      0
2           2   64  216.597983    5.274374      0
3           3   31  126.054855    9.584084      1
4           4   32  425.453296    2.578754      0
5           5   32   28.726570    5.072384      1
6           6   51  247.417987    8.660995      1
7           7   66  295.469928    9.253124      0
8           8   64  388.510288    9.242139      0
9           9   64   83.688539    5.403480      0


In [8]:
from ultralytics import YOLO
import torch

model = YOLO("/home/ubuntu/project/models/yolo/yolov8m.pt")

# GPU 확인
print(f"GPU 사용: {torch.cuda.is_available()}")

# 샘플 이미지로 예측 테스트
results = model.predict(
    source="/home/ubuntu/project/data/sample/images/",
    device=0,
    conf=0.25,
    save=False
)
print("✅ YOLO 예측 테스트 완료")
print(f"처리된 이미지 수: {len(results)}")

GPU 사용: True

image 1/10 /home/ubuntu/project/data/sample/images/sample_0.png: 640x640 (no detections), 9.6ms
image 2/10 /home/ubuntu/project/data/sample/images/sample_1.png: 640x640 (no detections), 9.5ms
image 3/10 /home/ubuntu/project/data/sample/images/sample_2.png: 640x640 (no detections), 9.5ms
image 4/10 /home/ubuntu/project/data/sample/images/sample_3.png: 640x640 (no detections), 9.5ms
image 5/10 /home/ubuntu/project/data/sample/images/sample_4.png: 640x640 (no detections), 9.5ms
image 6/10 /home/ubuntu/project/data/sample/images/sample_5.png: 640x640 (no detections), 9.5ms
image 7/10 /home/ubuntu/project/data/sample/images/sample_6.png: 640x640 (no detections), 9.5ms
image 8/10 /home/ubuntu/project/data/sample/images/sample_7.png: 640x640 (no detections), 9.5ms
image 9/10 /home/ubuntu/project/data/sample/images/sample_8.png: 640x640 (no detections), 9.5ms
image 10/10 /home/ubuntu/project/data/sample/images/sample_9.png: 640x640 (no detections), 9.5ms
Speed: 2.0ms preprocess, 

In [10]:
import torch
import timm

device = torch.device("cuda")

# num_classes=1000으로 로드 후 마지막 레이어만 교체
model = timm.create_model('efficientnet_b3', pretrained=False, num_classes=1000)
model.load_state_dict(torch.load(
    "/home/ubuntu/project/models/pretrained/efficientnet_b3.pth",
    weights_only=True  # 경고 제거
))

# 마지막 분류 레이어만 2클래스로 교체
import torch.nn as nn
model.classifier = nn.Linear(model.classifier.in_features, 2)
model = model.to(device)
model.eval()

# 샘플 테스트
dummy = torch.randn(4, 3, 224, 224).to(device)
with torch.no_grad():
    output = model(dummy)

print(f"✅ EfficientNet 테스트 완료")
print(f"출력 shape: {output.shape}")  # [4, 2] 나오면 정상

✅ EfficientNet 테스트 완료
출력 shape: torch.Size([4, 2])


In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("/home/ubuntu/project/data/sample/emr_sample.csv")

X = df.drop('label', axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

clf = XGBClassifier(device='cuda', n_estimators=10)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print(f"✅ XGBoost 테스트 완료")
print(f"Accuracy: {accuracy_score(y_test, pred):.2f}")

✅ XGBoost 테스트 완료
Accuracy: 0.67


/home/ubuntu/.conda/envs/team2/lib/python3.11/site-packages/xgboost/core.py:751: UserWarning: [11:28:36] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("/home/ubuntu/project/data/sample/emr_sample.csv")

X = df.drop('label', axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

# device 제거 → 경고 없어짐
clf = XGBClassifier(n_estimators=10)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print(f"✅ XGBoost 테스트 완료")
print(f"Accuracy: {accuracy_score(y_test, pred):.2f}")

✅ XGBoost 테스트 완료
Accuracy: 0.67


In [1]:
import timm

# Vision Transformer (최신 트렌드)
model_vit = timm.create_model('vit_base_patch16_224', pretrained=True)
print("✅ ViT-Base 완료")

# DenseNet (의료영상 강자)
model_dense = timm.create_model('densenet121', pretrained=True)
print("✅ DenseNet121 완료")

# ConvNeXt (최근 초음파에 많이 씀)
model_convnext = timm.create_model('convnext_base', pretrained=True)
print("✅ ConvNeXt 완료")

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

✅ ViT-Base 완료


model.safetensors:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

✅ DenseNet121 완료


model.safetensors:   0%|          | 0.00/354M [00:00<?, ?B/s]

✅ ConvNeXt 완료


In [2]:
code_utils = '''
import torch
import numpy as np
from sklearn.metrics import (
    roc_auc_score, f1_score, 
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns

# ================================
# 불균형 데이터 처리
# ================================
from imblearn.over_sampling import SMOTE

def balance_data(X, y):
    sm = SMOTE(random_state=42)
    X_res, y_res = sm.fit_resample(X, y)
    print(f"Before: {len(y)} → After: {len(y_res)}")
    return X_res, y_res

# ================================
# 의료 AI 핵심 지표
# ================================
def evaluate(y_true, y_pred, y_prob):
    print(classification_report(y_true, y_pred,
          target_names=["양성", "악성"]))
    print(f"AUC: {roc_auc_score(y_true, y_prob):.4f}")
    
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    print(f"민감도(Sensitivity): {tp/(tp+fn):.4f}")
    print(f"특이도(Specificity): {tn/(tn+fp):.4f}")

# ================================
# GradCAM 시각화
# ================================
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

def visualize_gradcam(model, img_tensor, target_layer):
    cam = GradCAM(model=model, target_layers=[target_layer])
    grayscale_cam = cam(input_tensor=img_tensor)
    return grayscale_cam

# ================================
# Early Stopping
# ================================
class EarlyStopping:
    def __init__(self, patience=10):
        self.patience = patience
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
    
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss >= self.best_loss:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

# ================================
# 학습 곡선 저장 (반출용)
# ================================
def plot_history(train_loss, val_loss, save_path):
    plt.figure(figsize=(8, 4))
    plt.plot(train_loss, label="Train Loss")
    plt.plot(val_loss, label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path)  # 반출 가능한 단순 선그래프
    print(f"저장완료: {save_path}")
'''

with open("/home/ubuntu/project/src/05_utils.py", "w") as f:
    f.write(code_utils)
print("✅ 05_utils.py 저장")

✅ 05_utils.py 저장


In [3]:
from ultralytics import YOLO

# YOLOv8 large (정확도 더 높음)
model_l = YOLO('yolov8l.pt')
print("✅ YOLOv8l 완료")

# YOLOv11 (최신버전)
model_11 = YOLO('yolo11m.pt')
print("✅ YOLO11m 완료")

✅ YOLOv8l 완료
✅ YOLO11m 완료


In [4]:
import os

# 반출 가능한 것들 저장 경로
os.makedirs("/home/ubuntu/project/download/models", exist_ok=True)
os.makedirs("/home/ubuntu/project/download/src", exist_ok=True)
os.makedirs("/home/ubuntu/project/download/graphs", exist_ok=True)

print("✅ 반출 폴더 준비완료")
print("""
반출 가능:
- download/models/ → 학습된 .pth 파일
- download/src/    → 소스코드 (100MB 이하)
- download/graphs/ → 단순 선그래프만
""")

✅ 반출 폴더 준비완료

반출 가능:
- download/models/ → 학습된 .pth 파일
- download/src/    → 소스코드 (100MB 이하)
- download/graphs/ → 단순 선그래프만

